In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True, timeout_ms=120000)

Mounted at /content/drive


In [2]:
import os

print(os.path.exists("/content/drive/MyDrive"))
print(os.listdir("/content/drive/MyDrive"))

path = "/content/drive/MyDrive/TP_NLP/dataset/raw/booktection/booktection_00000_A.txt"
print(os.path.exists(path))

True
['Git-2.39.2-64-bit.exe', 'IIA', 'Colab Notebooks', 'cambio conceptual e Innovación.pdf', 'Bloc de notas sin título (3).pdf', 'Copia de Talbot 2012 Bioethics Capítulo 4.pdf', 'CAPITULO 4.docx', 'CAPITULO 4.gdoc', '65598_Isidro_Valeriano_Entrega_final_domiciliaria_Isidro_Valeriano_559806_1256405549.pdf', 'Tercer Cuatri', 'ACSO-TP1-main', 'Documento sin título (3).gdoc', '1.Cinemática.note', '2. Series y sucesiones - Análisis Matemático III', 'isitosito', 'Guías (1).zip', '3. Señales discretas - Análisis Matemático III', 'Bloc de notas sin título (2).pdf', 'image (10).jpg', 'image (9).jpg', 'image (8).jpg', 'image (7).jpg', 'image (6).jpg', 'image (5).jpg', 'image (4).jpg', 'image (3).jpg', 'IMG-20230912-WA0000.jpg', 'IMG-20231009-WA0025.jpg', 'image (2).jpg', 'IMG-20231001-WA0035.jpg', 'IMG-20230925-WA0026.jpg', 'IMG-20230910-WA0049.jpg', 'IMG-20230810-WA0021.jpg', 'IMG_20230729_220757_200.jpg', '100_3468.JPG', 'IMG-20230515-WA0020.jpg', 'IMG-20230406-WA0068.jpg', 'imag

In [3]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path("/content/drive/MyDrive/TP_NLP")
DATASET_DIR = PROJECT_DIR / "dataset"
META_DIR = DATASET_DIR / "metadata"
DUALTEST_DIR = PROJECT_DIR / "DUALTEST"
RESULTS_DIR = DUALTEST_DIR / "results"

DUALTEST_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

(DUALTEST_DIR / "__init__.py").touch()

sys.path.append(str(PROJECT_DIR))

print("PROJECT_DIR:", PROJECT_DIR)
print("Existe proyecto:", PROJECT_DIR.exists())
print("DUALTEST_DIR:", DUALTEST_DIR)
print("Existe DUALTEST:", DUALTEST_DIR.exists())

PROJECT_DIR: /content/drive/MyDrive/TP_NLP
Existe proyecto: True
DUALTEST_DIR: /content/drive/MyDrive/TP_NLP/DUALTEST
Existe DUALTEST: True


In [4]:
!pip install transformers accelerate -q

In [5]:
import torch

print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Sin GPU: va a correr en CPU y puede tardar bastante.")

CUDA disponible: False
Sin GPU: va a correr en CPU y puede tardar bastante.


In [6]:
import importlib

import DUALTEST.metrics
import DUALTEST.text_utils
import DUALTEST.hf_model

importlib.reload(DUALTEST.metrics)
importlib.reload(DUALTEST.text_utils)
importlib.reload(DUALTEST.hf_model)

from DUALTEST.metrics import run_length_score, edit_similarity, token_overlap
from DUALTEST.text_utils import read_text, split_text_for_dualtest
from DUALTEST.hf_model import generate_completion

In [7]:
df_booktection = pd.read_csv(META_DIR / "booktection_metadata.csv")

df_originals = df_booktection[
    df_booktection["is_original"] == True
].copy()

print("BookTection completo:", df_booktection.shape)
print("Originales:", df_originals.shape)

print("\nMembership:")
print(df_originals["estimated_membership"].value_counts())

print("\nLengths:")
print(df_originals["length"].value_counts())

BookTection completo: (65656, 13)
Originales: (16414, 13)

Membership:
estimated_membership
member        10461
non_member     5953
Name: count, dtype: int64

Lengths:
length
medium    5495
small     5472
large     5447
Name: count, dtype: int64


In [8]:
sample = df_originals.iloc[0]

text = read_text(sample["file_path"])

prefix, continuation = split_text_for_dualtest(
    text,
    max_total_words=64,
    prefix_ratio=0.5
)

print("PREFIX\n")
print(prefix)

print("\n" + "="*100 + "\n")

print("GROUND TRUTH\n")
print(continuation)

print("\nPalabras prefix:", len(prefix.split()))
print("Palabras continuation:", len(continuation.split()))

PREFIX

O'Brien had sat down beside the bed, so that his face was almost on a level with Winston's. 'Three thousand,' he said, speaking over Winston's head to the man in the white


GROUND TRUTH

coat. Two soft pads, which felt slightly moist, clamped themselves against Winston's temples. He quailed. There was pain coming, a new kind of pain. O'Brien laid a hand reassuringly, almost kindly, on

Palabras prefix: 32
Palabras continuation: 32


In [9]:
import importlib
import DUALTEST.runner

importlib.reload(DUALTEST.runner)

from DUALTEST.runner import run_dualtest_dataframe

df_members = df_originals[
    df_originals["estimated_membership"] == "member"
].copy()

df_nonmembers = df_originals[
    df_originals["estimated_membership"] == "non_member"
].copy()

N = 100

df_eval = pd.concat([
    df_members.sample(N, random_state=42),
    df_nonmembers.sample(N, random_state=42)
]).sample(frac=1, random_state=42).reset_index(drop=True)

print(df_eval["estimated_membership"].value_counts())
print(df_eval["book_id"].value_counts().head(20))

estimated_membership
member        100
non_member    100
Name: count, dtype: int64
book_id
The_Housekeepers_-_Alex_Hay                            6
Happy_Place_-_Emily_Henry                              6
A_Game_of_Thrones_-_George_R._R._Martin                4
A_Spell_of_Good_Things_-_Ayobami_Adebayo               4
Love_Theoretically_-_Ali_Hazelwood                     4
Twilight_-_Stephenie_Meyer                             4
Youre_Not_Supposed_-_Kalynn_Bayron                     4
Highly_Suspicious_-_Talia_Hibbert                      4
The_Secret_Diary_of_Adrian_Mole_-_Sue_Townsend         3
Robyn_Harding_-_The_Drowning_Woman                     3
Cold_People_-_Tom_Rob_Smith                            3
The_Darkness_Before_Them_-_Matthew_Ward                3
The_Adventures_of_Tom_Sawyer_-_Mark_Twain              3
Watership_Down_-_Richard_Adams                         3
Anne_of_Green_Gables_-_L._M._Montgomery                3
All_the_Light_We_Cannot_See_-_Anthony_Doerr           

In [13]:
from huggingface_hub import HfApi
from google.colab import userdata
import os

# Obtener el token de Hugging Face de los secretos de Colab
HF_TOKEN = userdata.get('HF_token')


In [ ]:
df_results = run_dualtest_dataframe(
    df=df_eval,
    n=len(df_eval),
    model_name="Qwen/Qwen2.5-3B",
    max_total_words=64,
    prefix_ratio=0.5,
    max_new_tokens=64
)

Procesando: 0 booktection_09864_A.txt
Procesando: 1 booktection_08552_A.txt
Procesando: 2 booktection_07529_A.txt
Procesando: 3 booktection_11867_A.txt
Procesando: 4 booktection_15782_A.txt
Procesando: 5 booktection_10638_A.txt
Procesando: 6 booktection_04273_A.txt
Procesando: 7 booktection_13687_A.txt
Procesando: 8 booktection_14851_A.txt
Procesando: 9 booktection_00035_A.txt
Procesando: 10 booktection_10005_A.txt
Procesando: 11 booktection_16112_A.txt
Procesando: 12 booktection_13851_A.txt
Procesando: 13 booktection_02564_A.txt
Procesando: 14 booktection_10869_A.txt
Procesando: 15 booktection_16247_A.txt
Procesando: 16 booktection_00836_A.txt
Procesando: 17 booktection_14543_A.txt
Procesando: 18 booktection_06455_A.txt
Procesando: 19 booktection_04083_A.txt
Procesando: 20 booktection_14735_A.txt
Procesando: 21 booktection_03616_A.txt
Procesando: 22 booktection_10694_A.txt
Procesando: 23 booktection_07371_A.txt
Procesando: 24 booktection_10145_A.txt
Procesando: 25 booktection_00346_A.

In [ ]:
summary = df_results.groupby("estimated_membership")[[
    "run_length",
    "edit_similarity",
    "first_word_match",
    "token_overlap"
]].agg(["mean", "median", "std"])

display(summary)

In [ ]:
output_path = RESULTS_DIR / "dualtest_simple_qwen3b_100_member_100_nonmember.csv"
df_results.to_csv(output_path, index=False)

print("Guardado en:", output_path)